In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("incremental_flag","0")

In [0]:
print(dbutils.widgets.get("incremental_flag"))

create dimension table.

In [0]:
%sql
select * from parquet.`abfss://silver@pmngradls.dfs.core.windows.net/Clean Data/`;


fetch the records from the source table and join

In [0]:
%python
df_silver = spark.sql('''select distinct date_id from parquet.`abfss://silver@pmngradls.dfs.core.windows.net/Clean Data`''')

In [0]:
display(df_silver)


left join the data

In [0]:
if spark.catalog.tableExists("cars_catalog.gold.dim_date"):

    df_sink = spark.sql("select * from cars_catalog.gold.dim_date")

else:
    df_sink = spark.sql('''select 1 as dim_date_key,date_id from parquet.`abfss://silver@pmngradls.dfs.core.windows.net/Clean Data/` where 1 = 0''')

In [0]:
df_new_1 = df_silver.alias("df_silver").join(df_sink.alias("df_sink"),df_silver["date_id"] == df_sink["date_id"],"left").select(df_sink["dim_date_key"],df_silver["date_id"])

In [0]:
display(df_new_1)

old data

In [0]:
df_old = df_new_1.filter(col("dim_date_key").isNotNull())

display(df_old)

In [0]:
df_old = df_old.withColumn("dim_date_key", col("dim_date_key").cast("long"))

create model if not exists

In [0]:
df_silver.display()

new data

In [0]:
df_new = df_new_1.filter(col("dim_date_key").isNull()).select("date_id")
display(df_new)

create surrogate key

In [0]:
incremental_flag = dbutils.widgets.get("incremental_flag")

fetch the max surrograte key from the table.

In [0]:
if incremental_flag == "0":
    max_value = 0

else:
    max_value = spark.sql('''select max(dim_date_key) from cars_catalog.gold.dim_date''').collect()[0][0]

create new incremental keys

In [0]:
df_new.display()

In [0]:
df_new = df_new.withColumn("dim_date_key",max_value+monotonically_increasing_id()+1)
df_new.display()

In [0]:
df_new = df_new.select("dim_date_key","date_id")

In [0]:
# df_new = df_new.withColumn("dim_model_key",df_new["dim_model_key"].cast(IntegerType()))

In [0]:
display(df_new)

In [0]:
df_new.count()

In [0]:
df_old.count()

In [0]:
df_finals = df_new.union(df_old)

In [0]:
df_old.display()

In [0]:
df_finals.count()

SCD TYPE -1

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists("cars_catalog.gold.dim_date"):

    target_obj = DeltaTable.forPath(spark, "abfss://gold@pmngradls.dfs.core.windows.net/dim_date/")
    target_obj.alias("target").merge(df_finals.alias("source"),
                                     "target.dim_date_key = source.dim_date_key") \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
else:
    df_finals.write.format("delta").mode("overwrite").option("path","abfss://gold@pmngradls.dfs.core.windows.net/dim_date/")\
    .saveAsTable("cars_catalog.gold.dim_date")

In [0]:
%sql
select * from cars_catalog.gold.dim_date